In [ ]:
# ==========================================
# 0️⃣ Environment Setup
# ==========================================
# ⚠️ If running in Google Colab or a fresh virtual environment, 
# uncomment and run the lines below to install dependencies:

# ! If you are running this locally you will need atleast 15GB of VRAM to run the tests as they are written. If you have less than 15GB of VRAM, you can use a smaller model or a Free Google Collab instance with T4 GPU runtime.

# !pip install --upgrade pip
# !pip install setuptools==81.0.0
# !pip install -q transformers accelerate huggingface_hub
# !pip install -q exllamav2
# !pip install "numpy<2.0"

In [ ]:
# ==========================================
# 1️⃣ Model Definitions & Authentication
# ==========================================
import os
import gc
import math
import torch
import torch.nn.functional as F
from huggingface_hub import snapshot_download, login

# --- Configuration ---
# 13B Quantized Model (4-bit)
EXL2_ID = "LoneStriker/LLaMA2-13B-Estopia-4.0bpw-h6-exl2"

# 7B Full-Precision Baseline (16-bit)
# Chosen specifically to fit within a standard 15GB-16GB consumer VRAM limit
FP16_ID = "NousResearch/Llama-2-7b-hf"

# --- Smart Authentication ---
hf_token = None

# 1. Try to get token from Colab Secrets
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print("✅ Authenticated via Google Colab Secrets")
except ImportError:
    pass
except userdata.SecretNotFoundError:
    pass

# 2. Fallback to local environment variables
if not hf_token:
    hf_token = os.getenv("HF_TOKEN")
    if hf_token:
        print("✅ Authenticated via local environment variable")

if hf_token:
    login(token=hf_token)
    os.environ["HF_TOKEN"] = hf_token
else:
    print("⚠️ WARNING: No Hugging Face token found. Gated models may fail to download.")
    print("Please set your HF_TOKEN as a Colab Secret or local environment variable.")

# Download the EXL2 model (Transformers handles the FP16 model automatically)
print("\n📥 Verifying EXL2 checkpoint...")
exl2_path = snapshot_download(repo_id=EXL2_ID, cache_dir="./models")

In [ ]:
# ==========================================
# 2️⃣ Perplexity Calculation Functions
# ==========================================
def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()

def calculate_exl2_perplexity(text):
    from exllamav2 import ExLlamaV2, ExLlamaV2Config, ExLlamaV2Tokenizer
    
    config = ExLlamaV2Config()
    config.model_dir = exl2_path
    config.prepare()
    
    model = ExLlamaV2(config)
    model.load([16])
    tokenizer = ExLlamaV2Tokenizer(config)
    
    ids = tokenizer.encode(text)
    
    with torch.no_grad():
        logits = model.forward(ids, logits=True)
    
    logits = logits.cpu()
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = ids[..., 1:].contiguous()
    
    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)), 
        shift_labels.view(-1), 
        reduction="mean"
    )
    ppl = math.exp(loss.item())
    
    del model, tokenizer, config
    clear_vram()
    return ppl

def calculate_fp16_perplexity(text):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    
    tokenizer = AutoTokenizer.from_pretrained(FP16_ID)
    model = AutoModelForCausalLM.from_pretrained(
        FP16_ID, 
        torch_dtype=torch.float16, 
        device_map="auto"
    )
    
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss
        ppl = math.exp(loss.item())
        
    del model, tokenizer
    clear_vram()
    return ppl

In [ ]:
# ==========================================
# 3️⃣ Execution Loop 
# ==========================================
samples = [
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence is transforming many industries.",
    "Quantization reduces model size while preserving performance."
]

print("\n📊 Starting Quantized (13B) vs Unquantized (7B) Benchmark...\n")

for s in samples:
    print(f"Prompt: \"{s}\"")
    
    exl2_ppl = calculate_exl2_perplexity(s)
    print(f"  EXL2 (13B Quantized 4-bit)   -> PPL = {exl2_ppl:.2f}")
    
    fp16_ppl = calculate_fp16_perplexity(s)
    print(f"  FP16 (7B Unquantized 16-bit) -> PPL = {fp16_ppl:.2f}\n")

print("✅ Benchmarks complete!")